In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

from lakehouse_common import create_spark
spark = create_spark("Delivery - 05_serving_layer")

fact_df = spark.read.format("delta").load("s3a://lakehouse/warehouse/fact_delivery_task_gold")

n = fact_df.count()
print(f"Fact: {n} dòng")
if n == 0:
    raise ValueError("Fact rỗng - chạy lại Silver và Gold trước")

from lakehouse_common import require_unique_orders
require_unique_orders(fact_df)
fact_h = fact_df.withColumn("accept_hour", F.hour("accept_ts"))

# 1. Hiệu suất theo shipper
agg_courier = fact_df.groupBy("courier_id", "region_id").agg(
    F.count("order_id").alias("total_orders"),
    F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
    F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes"),
    F.round(F.max("delivery_minutes"), 2).alias("max_delivery_minutes")
).orderBy(F.desc("total_orders"))

# 2. Theo vùng và ngày (cho biểu đồ xu hướng)
agg_region_daily = fact_df.groupBy("region_id", "order_date").agg(
    F.count("order_id").alias("total_orders"),
    F.countDistinct("courier_id").alias("active_couriers"),
    F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
    F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes")
).orderBy("region_id", "order_date")

# 3. Theo giờ nhận đơn (cho biểu đồ khung giờ cao điểm)
agg_hourly = fact_h.groupBy("accept_hour").agg(
    F.count("order_id").alias("total_orders"),
    F.round(F.avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    F.round(F.expr("percentile_approx(delivery_minutes, 0.5)"), 2).alias("median_delivery_minutes")
).orderBy("accept_hour")

print("=== 5 SHIPPER CHẠY NHIỀU ĐƠN NHẤT ===")
agg_courier.show(5, truncate=False)

import os
jdbc = {
    "url": os.getenv("MYSQL_JDBC_URL", "jdbc:mysql://mysql:3306/delivery_source?allowPublicKeyRetrieval=true&useSSL=false"),
    "driver": "com.mysql.cj.jdbc.Driver",
    "user": os.getenv("MYSQL_USER", "delivery"),
    "password": os.getenv("MYSQL_PASSWORD", "delivery"),
    "truncate": "true",
}

for name, df in [("agg_courier_performance", agg_courier),
                 ("agg_region_daily", agg_region_daily),
                 ("agg_hourly", agg_hourly)]:
    df.coalesce(1).write.format("jdbc").options(**jdbc).option("dbtable", name).mode("overwrite").save()
    print(f"Đã đẩy bảng {name} sang MySQL ({df.count()} dòng)")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
import pyspark.sql.functions as F

B = "s3a://lakehouse/warehouse/"
bronze = spark.read.format("delta").load(B + "delivery_bronze") \
    .withColumn("event_date", F.to_date(
        F.concat(F.lit("2022-"), F.lpad(F.col("ds").cast("string"), 4, "0")), "yyyy-MMdd"))

cd = bronze.groupBy("courier_id", "event_date").agg(
    F.countDistinct("region_id").alias("n_region"),
    F.countDistinct("aoi_id").alias("n_aoi"))
print("courier-day            :", cd.count())
print("courier-day >1 region  :", cd.filter("n_region > 1").count())
print("courier-day >1 aoi     :", cd.filter("n_aoi > 1").count())

dim = spark.read.format("delta").load(B + "dim_courier_silver")
print("dim trùng (courier, effective_date):",
      dim.groupBy("courier_id", "effective_date").count().filter("count > 1").count())
overlap = dim.alias("a").join(dim.alias("b"),
    (F.col("a.courier_id") == F.col("b.courier_id")) &
    (F.col("a.effective_date") < F.col("b.effective_date")) &
    (F.col("b.effective_date") < F.col("a.end_date")))
print("dim chồng khoảng hiệu lực:", overlap.count())

fact = spark.read.format("delta").load(B + "fact_delivery_task_gold")
silver_reg = spark.read.format("delta").load(B + "delivery_silver") \
    .select("order_id", F.col("region_id").alias("order_region"))
mismatch = fact.join(silver_reg, "order_id").filter(F.col("region_id") != F.col("order_region")).count()
print(f"Fact.region_id lệch region của đơn: {mismatch} / {fact.count()}")